## Ishwor Chalise 
## 2408475

# Weather dataset

In [80]:
# Import all necessary libraries
import pymongo
from pprint import pprint
from bson.objectid import ObjectId
import string
import operator
import re
import json
import os

print("PyMongo Version:", pymongo.version)

#connect to mongodb
# Database connection configuration
yourUniId = "2408475"
url = "mongodb://localhost:27017/"
myDatabase = "db_" + yourUniId

# Create MongoDB client
myclient = pymongo.MongoClient(url)
print("Connection successful:", myclient)

# Connect to your database
mydb = myclient[myDatabase]
print("Connected to database:", mydb)

# Create collection reference
wcol = mydb["weather"]

# Test connection
myclient.admin.command('ping')
print("MongoDB server is running")


#Import weather data
# Check if data already exists
existing_count = wcol.count_documents({})
print(f"Existing documents: {existing_count}")

if existing_count == 0:
    print("Importing weather data...")
    
    # Method 1: Using mongoimport via command line
    # Uncomment and run in terminal:
    # mongoimport --db db_2408475 --collection weather --file weather.json
    
    # Method 2: Import using Python (if weather.json is in the same directory)
    try:
        with open('weather.json', 'r', encoding='utf-8') as file:
            # Check if it's a JSON array or JSONL format
            content = file.read()
            if content.startswith('['):
                data = json.loads(content)
                if isinstance(data, list):
                    wcol.insert_many(data)
                    print(f"Imported {len(data)} documents")
            else:
                # JSONL format (one JSON per line)
                file.seek(0)
                count = 0
                for line in file:
                    if line.strip():
                        doc = json.loads(line)
                        wcol.insert_one(doc)
                        count += 1
                print(f"Imported {count} documents")
    except FileNotFoundError:
        print("weather.json not found. Please ensure the file is in the current directory")
        print("Current directory:", os.getcwd())
    except Exception as e:
        print(f"Error importing data: {e}")
else:
    print("Data already exists in collection")

PyMongo Version: 4.16.0
Connection successful: MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True)
Connected to database: Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'db_2408475')
MongoDB server is running
Existing documents: 0
Importing weather data...
Imported 250 documents


## Basic Information

In [82]:
# Display collection information
print("\n=== COLLECTION INFORMATION ===")
print("Collections in", myDatabase, ":", mydb.list_collection_names())

# Count total documents
total_docs = wcol.count_documents({})
print("Total number of documents:", total_docs)

# Show sample document structure
print("\nSample document structure:")
sample = wcol.find_one()
if sample:
    pprint(sample)
else:
    print("No documents found. Please import data first.")




# Function to print tweets
def printTweets(tweetSet):
    """Iterate through and print tweets"""
    for tweet in tweetSet:
        pprint(tweet)

# Function to print update results
def printResult(result):
    """Print update operation results"""
    print('Match count:', result.matched_count)
    print('Modified count:', result.modified_count)
    print('Operation type:', type(result))

print("Helper functions defined successfully")


=== COLLECTION INFORMATION ===
Collections in db_2408475 : ['weather', 'newWeather', 'projWeather']
Total number of documents: 250

Sample document structure:
{'_id': ObjectId('69cce76435eca0cdfe6989b8'),
 'contributors': None,
 'coordinates': None,
 'created_at': 'Mon Feb 03 20:10:13 +0000 2020',
 'entities': {'hashtags': [{'indices': [36, 47], 'text': 'lionnation'},
                           {'indices': [48, 58], 'text': 'dasdpride'}],
              'media': [{'display_url': 'pic.twitter.com/5Bzh49ffAk',
                         'expanded_url': 'https://twitter.com/DASD_LMS/status/1224424872627122176/photo/1',
                         'id': 1224424871251398659,
                         'id_str': '1224424871251398659',
                         'indices': [83, 106],
                         'media_url': 'http://pbs.twimg.com/media/EP4H7uEX0AMHeoB.jpg',
                         'media_url_https': 'https://pbs.twimg.com/media/EP4H7uEX0AMHeoB.jpg',
                         'sizes': {'la

## Print Function

In [84]:
print("\n=== BASIC QUERIES ===")

# Find tweets from BBC Highlands
tweets = wcol.find({"user.name": "BBC Highlands"}, {"id": 1, "_id": 0})
print("\nTweets from BBC Highlands:")
printTweets(tweets)


=== BASIC QUERIES ===

Tweets from BBC Highlands:


## Pattern Matching

In [85]:
print("\n=== PATTERN MATCHING ===")

# Search for exact match "sun"
tweets = wcol.find({"text": "sun"}, {"text": 1})
print("\nExact 'sun' matches:")
printTweets(tweets)

# Regex search for "sun" (case-sensitive)
tweets = wcol.find({'text': {'$regex': 'sun'}}, {'text': 1})
print("\nCase-sensitive 'sun' matches:")
printTweets(tweets)

# Case-insensitive search for "sun"
tweets = wcol.find({'text': {'$regex': 'sun', '$options': 'i'}}, {'text': 1})
print("\nCase-insensitive 'sun' matches:")
printTweets(tweets)

# Count tweets with "rain"
count_rain = wcol.count_documents({'text': {'$regex': 'rain', '$options': 'i'}})
print("\nCount of tweets with 'rain':", count_rain)

# Count retweets (starting with RT)
count_rt = wcol.count_documents({'text': {'$regex': '^RT', '$options': 'i'}})
print("Count of retweets:", count_rt)

# Count tweets ending with "snow"
count_snow = wcol.count_documents({'text': {'$regex': 'snow$', '$options': 'i'}})
print("Count of tweets ending with 'snow':", count_snow)

# $in with regex (using re module)
tweets = wcol.find({'text': {'$in': [re.compile('.*sun.*', re.I), 
                                       re.compile('.*rain.*', re.I)]}}, 
                   {'text': 1})
print("\nTweets containing 'sun' or 'rain':")
count = 0
for tweet in tweets.limit(5):  # Show only first 5
    printTweets([tweet])
    count += 1
print(f"(Showing first {count} of many matches)")


=== PATTERN MATCHING ===

Exact 'sun' matches:

Case-sensitive 'sun' matches:
{'_id': ObjectId('69cce76435eca0cdfe6989be'),
 'text': 'I really might have seasonal depression because the sun is out?,  '
         'the weather is great, and I feel more motivated than ever.'}
{'_id': ObjectId('69cce76435eca0cdfe6989e5'),
 'text': 'Ever notice how the weather can impact our moods so much? A sunny '
         'day in Pittsburgh.. everyone is out &amp; about! The… '
         'https://t.co/rya6uN3oeI'}
{'_id': ObjectId('69cce76435eca0cdfe698a8d'),
 'text': 'RT @capitalweather: Ever hear the saying that a butterfly flapping '
         'its wings in China could bring rain instead of sunshine to New York? '
         'Tur…'}
{'_id': ObjectId('69cce76435eca0cdfe698a9d'),
 'text': 'RT @capitalweather: Ever hear the saying that a butterfly flapping '
         'its wings in China could bring rain instead of sunshine to New York? '
         'Tur…'}

Case-insensitive 'sun' matches:
{'_id': ObjectId('69c

## Operators, Null, Boolean

In [86]:
print("\n=== OPERATOR QUERIES ===")

# $gt: retweet count > 20
count_gt = wcol.count_documents({"retweeted_status.retweet_count": {'$gt': 20}})
print("Retweet count > 20:", count_gt)

# $eq: user.location is null
count_null = wcol.count_documents({"user.location": {'$eq': None}})
print("User location null:", count_null)

# $eq: user.geo_enabled is true
count_geo = wcol.count_documents({"user.geo_enabled": {'$eq': True}})
print("Geo enabled true:", count_geo)

# $lte: user.followers_count <= 10
count_followers = wcol.count_documents({"user.followers_count": {'$lte': 10}})
print("Followers <= 10:", count_followers)

# Combined query: find tweets with retweets > 10 AND from users with > 100 followers
combined = wcol.count_documents({
    "retweeted_status.retweet_count": {'$gt': 10},
    "user.followers_count": {'$gt': 100}
})
print("Retweets > 10 AND followers > 100:", combined)


=== OPERATOR QUERIES ===
Retweet count > 20: 33
User location null: 0
Geo enabled true: 117
Followers <= 10: 14
Retweets > 10 AND followers > 100: 29


# Aggregation (Group By)

In [87]:
print("\n=== AGGREGATION PIPELINE ===")

# Group by and calculate totals
grpBy = wcol.aggregate([
    {
        '$group': {
            '_id': None,
            'totalRetweetCount': {'$sum': "$retweeted_status.retweet_count"},
            'averageRetweetCount': {'$avg': "$retweeted_status.retweet_count"},
            'totalDocuments': {'$sum': 1},
            'maxRetweetCount': {'$max': "$retweeted_status.retweet_count"},
            'minRetweetCount': {'$min': "$retweeted_status.retweet_count"}
        }
    }
])

print("\nAggregation Results:")
for result in grpBy:
    pprint(result)

# Group by language
langGroup = wcol.aggregate([
    {
        '$group': {
            '_id': "$lang",
            'count': {'$sum': 1}
        }
    },
    {'$sort': {'count': -1}},
    {'$limit': 10}
])

print("\nTop 10 Languages:")
for result in langGroup:
    print(f"{result['_id']}: {result['count']} tweets")

# Group by user and count their tweets
userGroup = wcol.aggregate([
    {
        '$group': {
            '_id': "$user.screen_name",
            'tweetCount': {'$sum': 1},
            'avgFollowers': {'$avg': "$user.followers_count"}
        }
    },
    {'$sort': {'tweetCount': -1}},
    {'$limit': 10}
])

print("\nTop 10 Users by Tweet Count:")
for result in userGroup:
    print(f"@{result['_id']}: {result['tweetCount']} tweets, "
          f"Avg followers: {result['avgFollowers']:.0f}")


=== AGGREGATION PIPELINE ===

Aggregation Results:
{'_id': None,
 'averageRetweetCount': 25.697674418604652,
 'maxRetweetCount': 713,
 'minRetweetCount': 1,
 'totalDocuments': 250,
 'totalRetweetCount': 2210}

Top 10 Languages:
en: 249 tweets
pt: 1 tweets

Top 10 Users by Tweet Count:
@EVER_WEATHER: 3 tweets, Avg followers: 1003
@Hoodie_Weather_: 3 tweets, Avg followers: 729
@Septembers_Song: 2 tweets, Avg followers: 931
@Da_Coolest_Ever: 2 tweets, Avg followers: 678
@bIG_COn10: 2 tweets, Avg followers: 510
@michaelalfox: 1 tweets, Avg followers: 26527
@MichaelRPing1: 1 tweets, Avg followers: 84
@Colva1t: 1 tweets, Avg followers: 35
@Mari21_10: 1 tweets, Avg followers: 233
@HasanCelenk6: 1 tweets, Avg followers: 1205


## Cleaning Data – Update Operations

In [88]:
print("\n=== DATA CLEANING - LANGUAGES ===")

# Get distinct languages
distinct_langs = wcol.distinct("lang")
print("Distinct languages found:", distinct_langs[:10])  # Show first 10

# Language mapping dictionary
lang_map = {
    'en': 'English',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'it': 'Italian',
    'pt': 'Portuguese',
    'nl': 'Dutch',
    'ru': 'Russian',
    'ja': 'Japanese',
    'zh': 'Chinese',
    'und': 'Unknown'
}

# Find a sample document to update
sample_doc = wcol.find_one({'lang': {'$in': list(lang_map.keys())}})
if sample_doc:
    print(f"\nSample document ID: {sample_doc['_id']}")
    print(f"Current language code: {sample_doc.get('lang', 'Not set')}")
    
    # Update one document
    if sample_doc.get('lang') in lang_map:
        updateResult = wcol.update_one(
            {'_id': sample_doc['_id']}, 
            {'$set': {'lang': lang_map[sample_doc['lang']]}}
        )
        print("\nUpdate result:")
        printResult(updateResult)
        
        # Verify update
        updated = wcol.find_one({'_id': sample_doc['_id']}, {'lang': 1})
        print(f"Updated language: {updated.get('lang')}")

# Update all 'und' to 'Unknown'
print("\nUpdating 'und' language codes...")
updateResult = wcol.update_many(
    {'lang': "und"}, 
    {'$set': {'lang': 'Unknown'}}
)
printResult(updateResult)

# Update language codes based on mapping
for code, name in lang_map.items():
    if code not in ['und']:  # Skip already updated
        result = wcol.update_many(
            {'lang': code},
            {'$set': {'lang': name}}
        )
        if result.modified_count > 0:
            print(f"Updated {result.modified_count} documents from {code} to {name}")


=== DATA CLEANING - LANGUAGES ===
Distinct languages found: ['en', 'pt']

Sample document ID: 69cce76435eca0cdfe6989b8
Current language code: en

Update result:
Match count: 1
Modified count: 1
Operation type: <class 'pymongo.results.UpdateResult'>
Updated language: English

Updating 'und' language codes...
Match count: 0
Modified count: 0
Operation type: <class 'pymongo.results.UpdateResult'>
Updated 248 documents from en to English
Updated 1 documents from pt to Portuguese


### Clean Location Data

In [89]:
print("\n=== DATA CLEANING - LOCATIONS ===")

# Standardize location names
location_updates = {
    'England': 'UK',
    'UK': 'United Kingdom',
    'USA': 'United States',
    'US': 'United States',
    'America': 'United States'
}

for old_location, new_location in location_updates.items():
    result = wcol.update_many(
        {'user.location': {'$regex': old_location, '$options': 'i'}},
        {'$set': {'user.location': new_location}}
    )
    if result.modified_count > 0:
        print(f"Updated {result.modified_count} locations from '{old_location}' to '{new_location}'")

# Show updated locations
print("\nSample updated locations:")
tweets = wcol.find(
    {'user.location': {'$regex': 'United|UK', '$options': 'i'}},
    {'user.location': 1, '_id': 0}
).limit(5)
for tweet in tweets:
    print(tweet.get('user.location', 'No location'))


=== DATA CLEANING - LOCATIONS ===
Updated 2 locations from 'England' to 'UK'
Updated 9 locations from 'UK' to 'United Kingdom'
Updated 6 locations from 'USA' to 'United States'
Updated 13 locations from 'US' to 'United States'

Sample updated locations:
No location
No location
No location
No location
No location


### Remove Null Values

In [91]:
print("\n=== CLEANING NULL VALUES ===")

# Count documents with missing text
missing_text = wcol.count_documents({'text': None})
print(f"Documents with missing text: {missing_text}")

# Optionally remove documents with missing critical fields
if missing_text > 0:
    result = wcol.delete_many({'text': None})
    print(f"Deleted {result.deleted_count} documents with missing text")

# Fill null locations with 'Unknown'
result = wcol.update_many(
    {'user.location': None},
    {'$set': {'user.location': 'Unknown'}}
)
print(f"Updated {result.modified_count} documents with null location to 'Unknown'")


=== CLEANING NULL VALUES ===
Documents with missing text: 0
Updated 0 documents with null location to 'Unknown'


## Reshaping Data – Reduce Columns

In [92]:
print("\n=== REDUCING COLUMNS ===")

# Create new collection with selected fields only
wcol.aggregate([
    {
        '$project': {
            'user.location': 1,
            'text': 1,
            'created_at': 1,
            'lang': 1
        }
    },
    {'$out': "projWeather"}
])

# Verify new collection
projcol = mydb["projWeather"]
proj_count = projcol.count_documents({})
print(f"Original collection size: {total_docs} documents")
print(f"Projected collection size: {proj_count} documents")

# Show sample from projected collection
print("\nSample from projected collection:")
sample_proj = projcol.find_one()
if sample_proj:
    print("Keys in projected document:", list(sample_proj.keys()))


=== REDUCING COLUMNS ===
Original collection size: 250 documents
Projected collection size: 250 documents

Sample from projected collection:
Keys in projected document: ['_id', 'text', 'user', 'lang', 'created_at']


## Reduce Rows

In [93]:
print("\n=== REDUCING ROWS ===")

# Create new collection with limited rows
projcol.aggregate([
    {'$limit': 50},
    {'$out': "newWeather"}
])

# Check the new collection
newcol = mydb["newWeather"]
new_count = newcol.count_documents({})
print(f"Projected collection count: {proj_count}")
print(f"New collection count (limited): {new_count}")

# Verify it's a subset
if new_count > 0:
    print("\nFirst document in new collection:")
    printTweets([newcol.find_one()])


=== REDUCING ROWS ===
Projected collection count: 250
New collection count (limited): 50

First document in new collection:
{'_id': ObjectId('69cce76435eca0cdfe6989b8'),
 'created_at': 'Mon Feb 03 20:10:13 +0000 2020',
 'lang': 'English',
 'text': 'Best January bus duty weather EVER! #lionnation #dasdpride '
         'https://t.co/cTyq7ib3Z6 https://t.co/5Bzh49ffAk',
 'user': {'location': ''}}


## Further Analysis – Top Hashtags and Mentions

In [94]:
print("\n=== HASHTAG AND MENTION ANALYSIS ===")

def print_features(feature_set, feature_type):
    """Print top features"""
    print(f'\nTop 10 {feature_type}:')
    print('-' * 40)
    for item in feature_set:
        print(f"{item[0]}: {item[1]}")

def count_features(tweets):
    """Count hashtags and mentions in tweets"""
    tag_dict = {}
    mention_dict = {}
    
    for tweet in tweets:
        # Convert to string and lowercase
        tweet_text = str(tweet.get('text', ''))
        tweet_text = tweet_text.lower()
        words = tweet_text.split()
        
        for word in words:
            # Hashtags
            if word.startswith('#') and len(word) > 1:
                # Remove punctuation
                key = word.translate(str.maketrans('', '', string.punctuation))
                tag_dict[key] = tag_dict.get(key, 0) + 1
            
            # Mentions
            if word.startswith('@') and len(word) > 1:
                key = word.translate(str.maketrans('', '', string.punctuation))
                mention_dict[key] = mention_dict.get(key, 0) + 1
    
    # Get top 10
    top_tags = sorted(tag_dict.items(), key=operator.itemgetter(1), reverse=True)[:10]
    top_mentions = sorted(mention_dict.items(), key=operator.itemgetter(1), reverse=True)[:10]
    
    return top_tags, top_mentions

# Analyze tweets containing 'sun'
print("\nAnalyzing tweets mentioning 'sun':")
sun_tweets = newcol.find({'text': {'$regex': 'sun', '$options': 'i'}}, {'text': 1})
top_tags, top_mentions = count_features(sun_tweets)

print_features(top_tags, 'hashtags')
print_features(top_mentions, 'mentions')

# Analyze tweets containing 'rain'
print("\n\nAnalyzing tweets mentioning 'rain':")
rain_tweets = newcol.find({'text': {'$regex': 'rain', '$options': 'i'}}, {'text': 1})
top_tags, top_mentions = count_features(rain_tweets)

print_features(top_tags, 'hashtags')
print_features(top_mentions, 'mentions')

# Analyze tweets containing 'snow'
print("\n\nAnalyzing tweets mentioning 'snow':")
snow_tweets = newcol.find({'text': {'$regex': 'snow', '$options': 'i'}}, {'text': 1})
top_tags, top_mentions = count_features(snow_tweets)

print_features(top_tags, 'hashtags')
print_features(top_mentions, 'mentions')


=== HASHTAG AND MENTION ANALYSIS ===

Analyzing tweets mentioning 'sun':

Top 10 hashtags:
----------------------------------------

Top 10 mentions:
----------------------------------------


Analyzing tweets mentioning 'rain':

Top 10 hashtags:
----------------------------------------

Top 10 mentions:
----------------------------------------
replaycomic: 1
promotecomics: 1
topwebcomics: 1
webcomicupdates: 1
webcomicnetwork: 1


Analyzing tweets mentioning 'snow':

Top 10 hashtags:
----------------------------------------
snowfall: 1
climatechange: 1
weather…: 1

Top 10 mentions:
----------------------------------------


### Time-Based Analysis

In [96]:
print("\n=== TIME-BASED ANALYSIS ===")

# Aggregate by date (if created_at field exists)
date_analysis = wcol.aggregate([
    {
        '$group': {
            '_id': {'$substr': ["$created_at", 0, 10]},  # First 10 chars of date
            'count': {'$sum': 1},
            'avg_retweets': {'$avg': "$retweeted_status.retweet_count"}
        }
    },
    {'$sort': {'_id': 1}},
    {'$limit': 10}
])

print("\nTweets per date:")
for result in date_analysis:
    print(f"Date: {result['_id']}, Count: {result['count']}, "
          f"Avg retweets: {result['avg_retweets']:.2f}")


=== TIME-BASED ANALYSIS ===

Tweets per date:
Date: Mon Feb 03, Count: 250, Avg retweets: 25.70


### Sentiment Indicators Analysis

In [97]:
print("\n=== SENTIMENT INDICATORS ===")

# Define sentiment keywords
positive_words = ['good', 'great', 'awesome', 'amazing', 'nice', 'beautiful', 'sunny', 'wonderful']
negative_words = ['bad', 'terrible', 'awful', 'horrible', 'disaster', 'flood', 'storm', 'warning']

# Count tweets with positive indicators
pos_count = 0
for word in positive_words:
    pos_count += wcol.count_documents({'text': {'$regex': word, '$options': 'i'}})

# Count tweets with negative indicators
neg_count = 0
for word in negative_words:
    neg_count += wcol.count_documents({'text': {'$regex': word, '$options': 'i'}})

print(f"Tweets with positive words: {pos_count}")
print(f"Tweets with negative words: {neg_count}")

if pos_count + neg_count > 0:
    sentiment_ratio = pos_count / (pos_count + neg_count) * 100
    print(f"Positive sentiment ratio: {sentiment_ratio:.2f}%")


=== SENTIMENT INDICATORS ===
Tweets with positive words: 15
Tweets with negative words: 12
Positive sentiment ratio: 55.56%


## Export Results

In [100]:
print("\n=== EXPORTING RESULTS ===")

# Export top users to a JSON file
print("\nExporting top users...")
top_users = list(wcol.aggregate([
    {
        '$group': {
            '_id': "$user.screen_name",
            'tweetCount': {'$sum': 1}
        }
    },
    {'$sort': {'tweetCount': -1}},
    {'$limit': 20}
]))

# Convert ObjectId to string for JSON serialization
for user in top_users:
    if '_id' in user and isinstance(user['_id'], ObjectId):
        user['_id'] = str(user['_id'])

with open('top_users_2408475.json', 'w') as f:
    json.dump(top_users, f, indent=2)
print("Exported top users to 'top_users_2408475.json'")

# Export hashtag analysis
print("\nExporting hashtag analysis...")
hashtag_analysis = list(newcol.find({}, {'text': 1}).limit(1000))

# Convert ObjectId to string for JSON serialization
for doc in hashtag_analysis:
    if '_id' in doc and isinstance(doc['_id'], ObjectId):
        doc['_id'] = str(doc['_id'])

with open('hashtag_analysis_2408475.json', 'w') as f:
    json.dump(hashtag_analysis, f, indent=2)
print("Exported hashtag analysis to 'hashtag_analysis_2408475.json'")

print("\nExport complete!")


=== EXPORTING RESULTS ===

Exporting top users...
Exported top users to 'top_users_2408475.json'

Exporting hashtag analysis...
Exported hashtag analysis to 'hashtag_analysis_2408475.json'

Export complete!


# Summary 

In [101]:
print("\n=== FINAL SUMMARY STATISTICS ===")
print("=" * 50)

print(f"\nDatabase: {myDatabase}")
print(f"Total documents in weather collection: {total_docs}")
print(f"Documents in projected collection: {proj_count}")
print(f"Documents in new collection (50 limit): {new_count}")

# Collection sizes
print(f"\nCollections created:")
print(f"- projWeather: {proj_count} documents (reduced columns)")
print(f"- newWeather: {new_count} documents (reduced rows)")

# Data quality
null_counts = {
    'location_null': wcol.count_documents({'user.location': None}),
    'lang_unknown': wcol.count_documents({'lang': 'Unknown'}),
    'text_missing': wcol.count_documents({'text': None})
}
print(f"\nData quality metrics:")
for key, value in null_counts.items():
    print(f"- {key}: {value}")

# Cleaned counts
cleaned_langs = wcol.count_documents({'lang': {'$nin': ['en', 'es', 'fr', 'de', 'und']}})
print(f"- Languages standardized: {cleaned_langs}")

print("\n=== COURSEWORK COMPLETED ===")
print(f"Student: Ishwor Chalise (2408475)")
print(f"Date: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


=== FINAL SUMMARY STATISTICS ===

Database: db_2408475
Total documents in weather collection: 250
Documents in projected collection: 250
Documents in new collection (50 limit): 50

Collections created:
- projWeather: 250 documents (reduced columns)
- newWeather: 50 documents (reduced rows)

Data quality metrics:
- location_null: 0
- lang_unknown: 0
- text_missing: 0
- Languages standardized: 250

=== COURSEWORK COMPLETED ===
Student: Ishwor Chalise (2408475)
Date: 2026-04-01 15:41:28
